# Notebook #3: Feature Preprocessing
Some of the continuous features are transformed before they enter the correlation and regression analyses in the following notebooks. We check here each feature's distribution before and after its transformation, using two diagnostics:
- **Normality:** the Shapiro-Wilk test (`scipy.stats.shapiro`), which tests whether the distribution of a feature differs significantly from a normal distribution ($\alpha = 0.05$). With N = 234 the test flags even small departures from normality, so we read it together with the skewness.
- **Skewness:** the sample skewness $g_1$ (`scipy.stats.skew`, the Fisher-Pearson moment coefficient). We consider a skewness greater than 1 or less than -1 to indicate a highly skewed distribution, and values between -1 and 1 a moderately skewed one.

A transformation is applied on *distributional* grounds (it reduces skewness) or on *functional* grounds (it linearises the relationship with the outcome). We check the three continuous features used in the regression models as predictors, as well as two outcome variables:
- (predictor) Publication Age (weeks from publication to Dec 31st, 2025)
- (predictor) Number of Authors
- (predictor) Venue Impact (2-year mean citedness)
- (outcome) Citation Count (cumulative OpenAlex citations through Dec 31st, 2025) - corrected for zero-cited articles by adding 1 to all counts before log-transforming
- (outcome) Field-Weighted Citation Impact (FWCI) - corrected for zero-cited articles by applying the Dengler (2024) imputation method

The transformations proven to be effective are applied in the downstream notebooks, and the transformed values are used in the regression models (`build_feature_matrix`, `build_citations_frame`, `fwci_with_zeros_imputed`).


_References:_
- Clauset, A., Shalizi, C. R., & Newman, M. E. (2009). Power-law distributions in empirical data. SIAM review, 51(4), 661-703.
- Thelwall, M., & Wilson, P. (2014). Regression for citation data: An evaluation of different methods. Journal of Informetrics, 8(4), 963-971.
- Dengler, J. (2024). Determinants of citation impact. Vegetation Classification and Survey, 5, 169-177.


In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from helpers.config import VENUE_IMPACT_METRIC, VENUE_IMAPCT_METRIC_NAME
from helpers import dataset

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.


In [2]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


## Normality and Skewness Checks
`evaluate_transformation()` returns, for one feature, the Shapiro-Wilk statistic $W$ and p-value and the skewness $g_1$, both before and after the transformation. Each feature section below applies it, and the last section collects the results in one table.


In [3]:
def evaluate_transformation(
        feature: str, transformation: str, before: pd.Series | np.ndarray, after: pd.Series | np.ndarray
) -> pd.DataFrame:
    """One-row table of Shapiro-Wilk (W, p) and skewness g_1 of `feature`, before and after
    `transformation`. Missing values are dropped; `before` and `after` must stay the same length."""
    before, after = before.dropna(), after.dropna()
    if len(before) != len(after):
        raise ValueError(f"{feature}: {len(before)} values before the transformation, {len(after)} after")
    sw_before = stats.shapiro(before)
    w_before, p_before = float(sw_before.statistic), float(sw_before.pvalue)
    sw_after = stats.shapiro(after)
    w_after, p_after = float(sw_after.statistic), float(sw_after.pvalue)
    return pd.DataFrame(
        {
            "Transformation": transformation,
            "N": len(after),
            "W (before)": w_before,
            "p (before)": p_before,     # p < 0.05 indicates a significant departure from normality
            "g1 (before)": float(stats.skew(before)),
            "W (after)": w_after,
            "p (after)": p_after,       # p < 0.05 indicates a significant departure from normality
            "g1 (after)": float(stats.skew(after)),
        },
        index=pd.Index([feature], name="Feature"),
    )

## Feature Transformations


### Article Age
Publication age is $log$-transformed on *functional* rather than distributional grounds: citation accumulation is multiplicative, so a $log$-scale linearises the age-citation relationship.<br>
This follows the recommendations of `Thelwall and Wilson (2014)` and `Clauset et al. (2009)`.


In [4]:
weeks, _ = dataset.weeks_through_shared_year(combined)
age_summary = evaluate_transformation(
    "Publication Age (weeks)", "log(x)",
    before=weeks, after=np.log(weeks),
)
age_summary

,Transformation,N,W (before),p (before),g1 (before),W (after),p (after),g1 (after)
Feature,,,,,,,,
Publication Age (weeks),log(x),234,0.963982,0.000012,0.071523,0.960148,0.000004,-0.336953


### Citation Counts
Citation counts are $log$-transformed on *distributional* grounds: they are strongly right-skewed, with a few highly cited articles in a long tail, and the $log$-scale pulls that tail in. We add 1 to every count first, since uncited articles would otherwise map to $log(0)$.<br>
The count is the cumulative number of citations through the shared complete year (`dataset.citations_through_shared_year`), which is the outcome the citation models use, rather than the raw `TotalCitations`, whose census date differs between articles.<br>
This follows the recommendations of `Thelwall and Wilson (2014)` and `Clauset et al. (2009)`.


In [5]:
cumulative_citations, _ = dataset.citations_through_shared_year(combined)
citations_summary = evaluate_transformation(
    "Citation Count", "log(x + 1)",
    before=cumulative_citations,
    after=np.log(cumulative_citations + 1),
)
citations_summary

,Transformation,N,W (before),p (before),g1 (before),W (after),p (after),g1 (after)
Feature,,,,,,,,
Citation Count,log(x + 1),234,0.805289,2.131515e-16,1.956252,0.989207,0.077403,-0.262791


### Number of Authors
Number of authors is $log$-transformed on *distributional* grounds: author counts are right-skewed, with a few large collaborations in the tail. Every article has at least one author, so no offset is needed.<br>


In [6]:
authors_summary = evaluate_transformation(
    "Number of Authors", "log(x)",
    before=combined["NumAuthors"], after=np.log(combined["NumAuthors"]),
)
authors_summary

,Transformation,N,W (before),p (before),g1 (before),W (after),p (after),g1 (after)
Feature,,,,,,,,
Number of Authors,log(x),234,0.856024,5.396644e-14,1.38911,0.928181,2.978824e-09,0.141883


### Venue Impact Score
Venue impact score (two-year mean citedness) is kept in its natural scale on *distributional* grounds: it is only mildly right-skewed to begin with, and a $log$-transformation over-corrects it into a left-skewed distribution (the second table below).<br>


In [7]:
# log(2-year mean citedness) is rejected, shown here fow completeness; not used downstream
venue_summary = evaluate_transformation(
    f"Venue Impact ({VENUE_IMAPCT_METRIC_NAME})", "log(x), rejected",
    before=combined[VENUE_IMPACT_METRIC], after=np.log(combined[VENUE_IMPACT_METRIC]),
)
venue_summary

,Transformation,N,W (before),p (before),g1 (before),W (after),p (after),g1 (after)
Feature,,,,,,,,
Venue Impact (2-Year Mean Citedness),"log(x), rejected",234,0.929305,3.705403e-09,0.378255,0.850471,2.768824e-14,-0.965483


### FWCI
FWCI undergoes the Dengler (2024) preprocessing method to correct for zero-cited articles: each uncited article is given a FWCI score equal to half the minimum FWCI score of articles published within the same year, followed by a $log$-transformation to correct for skewness. This follows the recommendations of `Dengler (2024)`.


In [8]:
fwci_summary = evaluate_transformation(
    "FWCI", "Dengler imputation",
    before=combined["FieldWeightedCitationIndex"],
    after=np.log(dataset.fwci_with_zeros_imputed(combined)),
)
fwci_summary

,Transformation,N,W (before),p (before),g1 (before),W (after),p (after),g1 (after)
Feature,,,,,,,,
FWCI,Dengler imputation,234,0.849974,2.610718e-14,1.438401,0.974537,0.000321,-0.470501


## Summary
All five features, before and after their transformation.


In [9]:
pd.concat([age_summary, citations_summary, authors_summary, venue_summary, fwci_summary])

,Transformation,N,W (before),p (before),g1 (before),W (after),p (after),g1 (after)
Feature,,,,,,,,
Publication Age (weeks),log(x),234,0.963982,1.234925e-05,0.071523,0.960148,4.251831e-06,-0.336953
Citation Count,log(x + 1),234,0.805289,2.131515e-16,1.956252,0.989207,7.740338e-02,-0.262791
Number of Authors,log(x),234,0.856024,5.396644e-14,1.389110,0.928181,2.978824e-09,0.141883
Venue Impact (2-Year Mean Citedness),"log(x), rejected",234,0.929305,3.705403e-09,0.378255,0.850471,2.768824e-14,-0.965483
FWCI,Dengler imputation,234,0.849974,2.610718e-14,1.438401,0.974537,3.208970e-04,-0.470501
